In [1]:
%%capture
!pip install --upgrade transformers bitsandbytes accelerate peft datasets wandb contractions spacy tqdm

In [2]:
import torch
import wandb
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset, DatasetDict
import numpy as np
import re
from sklearn.metrics import precision_score, f1_score, recall_score, accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from contractions import fix
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tqdm import tqdm
from tqdm.contrib.concurrent import process_map 

# Login to Weights & Biases (W&B) - Required in Kaggle
wandb.login(key="REDACTED")  # Replace with your actual API key
wandb.init(project="Spring2025")


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: dhruvjoshi892000 (dhruvjoshi892000-pace-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:
# Ensure GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} for training")

Using cuda for training


In [4]:
import spacy

# Load spaCy model with GPU support
# spacy.require_gpu()  # Ensure spaCy runs on GPU
nlp = spacy.load("en_core_web_sm")

# Verify if GPU is used
print("spaCy is using GPU:", spacy.prefer_gpu())


# Load spaCy's English NLP model
nlp = spacy.load("en_core_web_sm")

def preprocess_text(text):
    """
    Preprocess legal text for analysis using spaCy.
    Steps:
    - Lowercasing
    - Removing HTML tags
    - Removing URLs
    - Expanding contractions
    - Removing special characters
    - Removing numbers
    - Removing stop words
    - Lemmatization using spaCy
    - Removing extra whitespace
    """
    # Lowercase the text
    text = text.lower()
    
    # Remove HTML tags and URLs
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    
    # Expand contractions (optional)
    try:
        text = fix(text)
    except ImportError:
        pass  # Skip if the 'contractions' package is not available
    
    # Remove special characters and numbers (excluding spaces)
    text = re.sub(r'[^a-z\s]', '', text)

    # Process text using spaCy
    doc = nlp(text)
    
    # Lemmatize and remove stopwords
    text = ' '.join([token.lemma_ for token in doc if not token.is_stop])

    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text


# Define tokenization function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

# Load dataset from CSV
df = pd.read_csv("/kaggle/input/15_labels_data.csv")  # Adjust file path if needed

# Apply preprocessing
# df = df[:10]
# df["text"] = df["text"].apply(preprocess_text)
processed_texts = [preprocess_text(text) for text in tqdm(df['text'], desc="Preprocessing texts")]

# Convert DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Split dataset into train and validation
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset['train']
eval_dataset = dataset['test']


# Load tokenizer
model_name = "nlpaueb/legal-bert-base-uncased"  # Modify as needed
tokenizer = AutoTokenizer.from_pretrained(model_name)


# Apply tokenization
train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)

/usr/local/lib/python3.10/dist-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.7.1) was trained with spaCy v3.7.2 and may not be 100% compatible with the current version (3.8.4). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


spaCy is using GPU: True


Preprocessing texts: 100%|██████████| 8417/8417 [34:17<00:00,  4.09it/s]  


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/7575 [00:00<?, ? examples/s]

Map:   0%|          | 0/842 [00:00<?, ? examples/s]

In [5]:
# Set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# Load pre-trained model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=15).to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    accuracy = accuracy_score(labels, predictions)
    balanced_acc = balanced_accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average='weighted', zero_division=0)
    recall = recall_score(labels, predictions, average='weighted', zero_division=0)
    f1 = f1_score(labels, predictions, average='weighted', zero_division=0)
    return {
        "accuracy": accuracy,
        "balanced_accuracy": balanced_acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

training_args = TrainingArguments(
    output_dir="/kaggle/working/results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="/kaggle/working/logs",
    label_names=["labels"],
    logging_steps=10,
    num_train_epochs=15,  # More epochs for better legal text learning
    per_device_train_batch_size=4,  # Lower batch size for stability
    per_device_eval_batch_size=4,
    learning_rate=2e-5,  # Reduce LR for better fine-tuning
    warmup_ratio=0.1,  # Stabilize early training
    weight_decay=0.01,
    metric_for_best_model="balanced_accuracy",
    load_best_model_at_end=True,
    report_to=["wandb"],  # Enable W&B logging
    fp16=torch.cuda.is_available(),  # Use FP16 if GPU is available
)


pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [6]:
import random
# 🔧 Set Random Seed for Reproducibility
random.seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

In [7]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics
)

# Start training
trainer.train()

# Evaluate the model
results = trainer.evaluate()
print(results)


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Balanced Accuracy,Precision,Recall,F1
1,1.084500,0.997861,0.711401,0.424786,0.736141,0.711401,0.694550
2,0.495500,0.782068,0.785036,0.613664,0.774386,0.785036,0.777027
3,0.324400,0.774579,0.790974,0.656460,0.793961,0.790974,0.789474
4,0.486600,1.034306,0.783848,0.662559,0.778974,0.783848,0.778628
5,0.320200,1.638175,0.788599,0.663591,0.787813,0.788599,0.787031
6,0.648500,3.005885,0.769596,0.617262,0.780133,0.769596,0.769787
7,0.168600,3.367031,0.785036,0.653529,0.786555,0.785036,0.781971
8,0.799300,3.689400,0.770784,0.644269,0.780369,0.770784,0.773018
9,0.016900,3.533869,0.777910,0.664793,0.782145,0.777910,0.777207
10,0.346400,3.499251,0.792162,0.685814,0.786799,0.792162,0.787647


/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked t

{'eval_loss': 3.5530104637145996, 'eval_accuracy': 0.7933491686460807, 'eval_balanced_accuracy': 0.6869104592722886, 'eval_precision': 0.7922434587926477, 'eval_recall': 0.7933491686460807, 'eval_f1': 0.791525021331554, 'eval_runtime': 18.6807, 'eval_samples_per_second': 45.073, 'eval_steps_per_second': 5.674, 'epoch': 15.0}


In [8]:
# 279 labels

In [9]:
import spacy

# Load spaCy model with GPU support
# spacy.require_gpu()  # Ensure spaCy runs on GPU
nlp = spacy.load("en_core_web_sm")

# Verify if GPU is used
print("spaCy is using GPU:", spacy.prefer_gpu())


# Load spaCy's English NLP model
nlp = spacy.load("en_core_web_sm")

def preprocess_text(text):
    """
    Preprocess legal text for analysis using spaCy.
    Steps:
    - Lowercasing
    - Removing HTML tags
    - Removing URLs
    - Expanding contractions
    - Removing special characters
    - Removing numbers
    - Removing stop words
    - Lemmatization using spaCy
    - Removing extra whitespace
    """
    # Lowercase the text
    text = text.lower()
    
    # Remove HTML tags and URLs
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    
    # Expand contractions (optional)
    try:
        text = fix(text)
    except ImportError:
        pass  # Skip if the 'contractions' package is not available
    
    # Remove special characters and numbers (excluding spaces)
    text = re.sub(r'[^a-z\s]', '', text)

    # Process text using spaCy
    doc = nlp(text)
    
    # Lemmatize and remove stopwords
    text = ' '.join([token.lemma_ for token in doc if not token.is_stop])

    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text


# Define tokenization function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

# Load dataset from CSV
df = pd.read_csv("/kaggle/input/279_labels_data.csv")  # Adjust file path if needed

# Apply preprocessing
# df = df[:10]
# df["text"] = df["text"].apply(preprocess_text)
processed_texts = [preprocess_text(text) for text in tqdm(df['text'], desc="Preprocessing texts")]

# Convert DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Split dataset into train and validation
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset['train']
eval_dataset = dataset['test']


# Load tokenizer
model_name = "nlpaueb/legal-bert-base-uncased" # Modify as needed
tokenizer = AutoTokenizer.from_pretrained(model_name)


# Apply tokenization
train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)

/usr/local/lib/python3.10/dist-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.7.1) was trained with spaCy v3.7.2 and may not be 100% compatible with the current version (3.8.4). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
/usr/local/lib/python3.10/dist-packages/spacy/util.py:1740: UserWarning: [W111] Jupyter notebook detected: if using `prefer_gpu()` or `require_gpu()`, include it in the same cell right before `spacy.load()` to ensure that the model is loaded on the correct device. More information: http://spacy.io/usage/v3#jupyter-notebook-gpu
  warnings.warn(Warnings.W111)


spaCy is using GPU: True


Preprocessing texts: 100%|██████████| 8417/8417 [35:40<00:00,  3.93it/s]  


Map:   0%|          | 0/7575 [00:00<?, ? examples/s]

Map:   0%|          | 0/842 [00:00<?, ? examples/s]

In [10]:
# Set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# Load pre-trained model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=279).to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    accuracy = accuracy_score(labels, predictions)
    balanced_acc = balanced_accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average='weighted', zero_division=0)
    recall = recall_score(labels, predictions, average='weighted', zero_division=0)
    f1 = f1_score(labels, predictions, average='weighted', zero_division=0)
    return {
        "accuracy": accuracy,
        "balanced_accuracy": balanced_acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

training_args = TrainingArguments(
    output_dir="/kaggle/working/results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="/kaggle/working/logs",
    label_names=["labels"],
    logging_steps=10,
    num_train_epochs=15,  # More epochs for better legal text learning
    per_device_train_batch_size=4,  # Lower batch size for stability
    per_device_eval_batch_size=4,
    learning_rate=2e-5,  # Reduce LR for better fine-tuning
    warmup_ratio=0.1,  # Stabilize early training
    weight_decay=0.01,
    metric_for_best_model="balanced_accuracy",
    load_best_model_at_end=True,
    report_to=["wandb"],  # Enable W&B logging
    fp16=torch.cuda.is_available(),  # Use FP16 if GPU is available
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [11]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics
)

# Start training
trainer.train()

# Evaluate the model
results = trainer.evaluate()
print(results)


/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Balanced Accuracy,Precision,Recall,F1
1,4.665100,4.568449,0.210214,0.055618,0.114599,0.210214,0.112116
2,2.761000,3.130279,0.440618,0.206097,0.332064,0.440618,0.344791
3,2.095000,2.571218,0.502375,0.280926,0.394183,0.502375,0.422667
4,1.837200,2.307489,0.541568,0.327651,0.457556,0.541568,0.472399
5,1.950300,2.220852,0.555819,0.375367,0.514204,0.555819,0.509469
6,1.293700,2.198902,0.576010,0.390879,0.530776,0.576010,0.530152
7,0.857900,2.181390,0.579572,0.407715,0.547774,0.579572,0.541676
8,0.816400,2.149887,0.586698,0.419163,0.572490,0.586698,0.558832
9,0.782200,2.148621,0.583135,0.417227,0.563845,0.583135,0.554338
10,0.676900,2.128056,0.608076,0.456590,0.590334,0.608076,0.581850


/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:2184: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all 

{'eval_loss': 2.2422006130218506, 'eval_accuracy': 0.6247030878859857, 'eval_balanced_accuracy': 0.5085836344374459, 'eval_precision': 0.6170024154216811, 'eval_recall': 0.6247030878859857, 'eval_f1': 0.6026301271927091, 'eval_runtime': 18.8616, 'eval_samples_per_second': 44.641, 'eval_steps_per_second': 5.62, 'epoch': 15.0}


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:2184: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
